<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## ➕ LangGraph Reducer 全景速查

**一句话定位**:state 每个 key 各有一个 reducer——`new = reducer(left=旧值, right=节点返回的增量)`;**没写 reducer = 覆盖**。
> 以下全部在 `langgraph 1.2.0` / `langchain-core 1.4.0` 实测;文档来源:LangGraph Graph API 官方文档。

</div>

### 1️⃣ 核心规则

| 规则 | 说明 |
|---|---|
| **每个 key 独立** | 各自一个 reducer,互不影响 |
| **节点返回增量** | 只返回要改的 key;没返回的 key **原样保留** |
| **默认 = 覆盖** | 无 `Annotated` → 后写覆盖先写 |
| **参数顺序** | `reducer(left=旧值, right=新增量)`,输入 `invoke(...)` 也会经过 reducer(`left` 初始为空值,如 `[]`) |
| **合并时机** | 每个 **super-step 结束**时,把该步所有节点的增量统一并入 |
| **节点返回 `{}` / `None`** | 无更新,state 不变 |
| **适用 schema** | `TypedDict` · `dataclass` · `Pydantic BaseModel` 均支持 `Annotated` |

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    plain: int                                   # 无 reducer → 覆盖
    items: Annotated[list[int], operator.add]    # 有 reducer → 拼接

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅰️ 第一部分:无 reducer(默认覆盖)

</div>

| 情形 | 结果 |
|---|---|
| 单节点返回 `{"foo": 2}`(原 `foo=1`) | `foo = 2`(覆盖);其他 key 不动 |
| **并行**节点同一步写同一个 key | ❌ `InvalidUpdateError: Can receive only one value per step. Use an Annotated key to handle multiple values.` |
| 并行节点写**不同** key | ✅ 无冲突 |
| 串行节点(不同 step)先后写同一 key | ✅ 后者覆盖前者 |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅱️ 第二部分:内置 / 常用 reducer

</div>

#### 2.1 `operator.*` 与普通二元函数

| reducer | 类型 | 行为(实测) |
|---|---|---|
| `operator.add` | `list` | 拼接 `[0]+[1]→[0,1]` ✅ 最常用 |
| `operator.add` | `int` | 累加 `1+5→6` |
| `operator.add` | `str` | 字符串拼接 `"a"+"x"→"ax"` |
| `operator.or_` | `dict` | 合并(同 key 后者覆盖) |
| `operator.or_` | `set` | 并集 |
| `lambda a,b: max(a,b)` | `int` | 取大值 |
| 自定义函数 | 任意 | 见第三部分 |

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**坑 1**:不能直接写 `Annotated[int, max]`——内置 `max` 无法取签名,报 `ValueError: no signature found for builtin`。**用 `lambda a, b: max(a, b)` 或自定义 `def`**。

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**坑 2**:`operator.add` 要求**增量类型匹配**。字段是 list,节点返回裸值 `2` → `TypeError: can only concatenate list (not "int")`。**必须返回 `[2]`**。

</div>

#### 2.2 初始值缺失 / `None`

| 输入 | `Annotated[list, add]` | 自定义 `(l or []) + (r or [])` |
|---|---|---|
| `invoke({})`(没传该 key) | ✅ 视为空 `[]` | ✅ |
| `invoke({"foo": None})` | ❌ `TypeError: ... concatenate list (not "NoneType")` | ✅ 自定义里处理 `None` |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅲 第三部分:自定义 reducer

</div>

In [ ]:
def merge_dict(left: dict | None, right: dict | None) -> dict:
    return {**(left or {}), **(right or {})}      # 防御 None

def keep_last_n(left: list | None, right: list | None) -> list:
    return ((left or []) + (right or []))[-5:]    # 滑动窗口

class State(TypedDict):
    d: Annotated[dict, merge_dict]
    window: Annotated[list, keep_last_n]

**写法要点**
- 签名固定 `(left, right) -> new`,**返回值**成为新 state
- 一定处理 `None`(初始值或 `invoke({"k": None})`)
- 实测:并行节点 `a/b/c` 各返回 `["a"]/["b"]/["c"]` → 结果 `['a','b','c']`(按节点顺序合并,但**不要依赖顺序**)
- 并行节点**互相看不到**对方本 step 的写入——都读同一份旧 state

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅳 第四部分:清空 / 绕过 reducer —— `Overwrite`

</div>

In [ ]:
from langgraph.types import Overwrite

def reset(state):
    return {"items": Overwrite([])}          # 绕过 reducer,直接替换
    # 等价 dict 写法:{"items": {"__overwrite__": []}}

| 情形 | 结果(实测) |
|---|---|
| 返回 `{"items": []}` | ❌ **不会清空**(`[]` 被拼进去,旧值仍在) |
| 返回 `{"items": Overwrite([])}` | ✅ 清空为 `[]` |
| 返回 `{"items": {"__overwrite__": [7]}}` | ✅ 同 `Overwrite([7])` |
| 同一 step **两个**节点各返回 `Overwrite` | ❌ `InvalidUpdateError: Can receive only one Overwrite value per super-step` |
| 同一 step 一个 `Overwrite` + 一个普通增量 | `Overwrite` 生效(实测 `[9]`,普通增量被丢弃) |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:`Overwrite` 只对**带 reducer 的 key**有意义(对 `BinaryOperatorAggregate` channel);无 reducer 的 key 本来就是覆盖。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅴 第五部分:`add_messages`(消息专用 reducer)

</div>

In [ ]:
from typing import Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph import MessagesState, add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]   # 手写

class State2(MessagesState):                              # 等价,内置 messages 字段
    extra: str

#### 行为全表(`add_messages(left, right)`,均实测)

| 情形 | 结果 |
|---|---|
| 新消息 id 不存在 | **追加** |
| 新消息 **id 与已有相同** | **替换**该条(位置不变) |
| 新消息没 id | **自动生成 id** |
| `right` 是单个 Message | ✅ 自动当成 `[msg]` |
| `right` 含 `("human","t")` / `{"role":"assistant","content":"d"}` / `"s"` | 转成 `HumanMessage` / `AIMessage` / `HumanMessage` |
| `RemoveMessage(id="1")` | 删除该条 |
| `RemoveMessage(id="不存在")` | ❌ `ValueError: Attempting to delete a message with an ID that doesn't exist` |
| `RemoveMessage(id=REMOVE_ALL_MESSAGES)` + 新消息 | **清空历史**,只剩其后的新消息 |
| `format="langchain-openai"` | 内容规范成 OpenAI 格式(需 `langchain-core>=0.3.11`) |
| 直接调用 `add_messages(None, [...])` | ❌ `ValueError: Must specify non-null arguments for both 'left' and 'right'`(作为 channel 用时框架自动给空 `left`) |

In [ ]:
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES

def trim(state):
    return {"messages": [RemoveMessage(id=m.id) for m in state["messages"][:-2]]}

def wipe(state):
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES)]}

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**读取注意**:`add_messages` 会把 dict/tuple 反序列化成 Message 对象,节点里要用**属性访问** `state["messages"][-1].content`,不是 `["content"]`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅵 第六部分:与其他机制配合

</div>

| 机制 | reducer 行为(实测) |
|---|---|
| **`Command(update=..., goto=...)`** | `update` 同样走 reducer(`["cmd"]` + 后续节点 `["b"]` → `['cmd','b']`) |
| **`Send` 并行扇出(map-reduce)** | 各 worker 返回的增量全部汇入带 reducer 的 key:`['A','B','C']`;**没 reducer 则 `InvalidUpdateError`** |
| **`input_schema` / `output_schema`** | reducer 声明在 **总 state** 上;输入 `{"foo":[1]}` → 输出 `{"foo":[1,2]}`,`hidden` 等私有 key 不出现在 `invoke` 结果 |
| **dataclass / Pydantic state** | `Annotated[list, operator.add]` 同样有效;节点用属性访问 `state.foo` |
| **并行 + `add_messages`** | 两个分支的消息都被追加:`['q','B','C']` |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**`Command` 陷阱**(官方文档):`Command` 只加**动态边**,该节点的静态边照样执行,一个节点用一种路由机制。仅用 `Command(update=...)` 作为 **输入** 可能让已结束的图"卡住";续聊请传普通 input dict。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅶 第七部分:新特性 `DeltaChannel`(beta)

</div>

In [ ]:
from langgraph.channels import DeltaChannel

def reducer(state: list | None, writes: list[list]) -> list:    # 注意:收到的是「一批」writes
    out = list(state or [])
    for w in writes:
        out.extend(w)
    return out

class State(TypedDict):
    foo: Annotated[list, DeltaChannel(reducer)]        # snapshot_frequency 默认 1000

| 项 | 内容 |
|---|---|
| 目的 | checkpoint 只存**增量/哨兵**,靠重放祖先 writes 重建 state,**省存储**(长对话) |
| 签名 | `reducer(state, [write1, write2, …]) -> new_state`(**批量**,与普通 `(left,right)` 不同) |
| 约束 | 必须**确定性**且**批次不变性**:`r(r(s,xs),ys) == r(s, xs+ys)` |
| `snapshot_frequency` | 每 N 次更新存一次完整快照(默认 1000) |
| 实测 | 两个串行节点 `[2]`、`[3]` → `[1, 2, 3]` ✅ |
| ⚠️ 状态 | **Beta**:API 与落盘格式可能变,官方文档 Graph API 页未收录 |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📋 决策速查

</div>

| 想要 | 用 |
|---|---|
| 覆盖(最新值为准) | 不写 reducer |
| 日志 / 结果收集 / map-reduce | `Annotated[list, operator.add]` |
| 计数 / 累加 | `Annotated[int, operator.add]` |
| 合并 dict / set | `operator.or_` 或自定义 |
| 取 max / 去重 / 滑窗 | 自定义 `(left, right)` 函数(处理 `None`) |
| 对话消息 | `add_messages` / `MessagesState` |
| 删 / 清消息 | `RemoveMessage(id=...)` / `REMOVE_ALL_MESSAGES` |
| 清空带 reducer 的字段 | `Overwrite(value)` |
| 超长对话省存储 | `DeltaChannel`(beta) |

#### 排错对照

| 报错 | 原因 → 解法 |
|---|---|
| `InvalidUpdateError: Can receive only one value per step` | 并行写同 key 没 reducer → 加 `Annotated` |
| `InvalidUpdateError: only one Overwrite value per super-step` | 同步多个 `Overwrite` → 只留一个 |
| `TypeError: can only concatenate list` | 增量不是 list → 包 `[x]`;或输入 `None` → 自定义 reducer |
| `ValueError: no signature found for builtin` | 用了 `max`/`min` → 改 lambda |
| `ValueError: ... ID that doesn't exist` | `RemoveMessage` 的 id 不在 state 里 |

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**:reducer = `(旧值, 增量) -> 新值`,每个 key 一个、step 末统一合并;**无 reducer 覆盖、并行必须有**;list 用 `add`、消息用 `add_messages`、清空用 `Overwrite`、自定义函数记得处理 `None`。

</div>